# Stage 4 — per-design 3D topology optimization of the top-10 blades

Runs **frozen-aero-skin SIMP TO on each** of the top-10 designs (ranked by the fine 3D
`J_fan` from Stage-3.C), trimming mass by hollowing the rib cores + thick-panel interiors
while holding the air-pushing panel faces solid. **Screened** mode finds, per design, the most
material removed that still passes the structural screen. Output: a carved density field per
design + a `summary.json` with the mass removed and the screen result. Then choose **3** to print.

Logic lives in `fanopt.topopt.blade_topopt` + `scripts/run_phase2_blade_to.py`; this notebook
only orchestrates. Runtime depends on mesh + parallelism (see the fidelity guide in cell 3).


## 1. Connect Drive

In [ ]:
# Drive connect ONLY (kept separate from the repo/deps install below).
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = Path.cwd() / "data"
print("drive root:", DRIVE_ROOT)

## 2. Repo + deps  (separate cell from the Drive connect above)

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the TO tool + this notebook land on main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    # TO stack: gmsh + CadQuery for the solid mesh, scikit-fem for the 3D FEA.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gmsh", "cadquery", "scikit-fem"], check=True)
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO)

## 3. Config — point at the campaign + set the TO fidelity
**Screened mode** (default): per design, find the most aggressive (lowest-volfrac) TO that
still passes the structural screen — max material removed while keeping integrity. **Finer
mesh** resolves internal beam structure in the thick ribs and lets the frozen skin be thinner;
cost scales steeply (see the mesh guide below). Needs a high-RAM session (the L4 >50 GB is ideal).

In [ ]:
from fanopt.topopt.blade_topopt import DEFAULT_VOLFRAC_LADDER

# These paths already point at the Stage-3 campaign + the Stage-3.C verification output — the
# exact folders the earlier notebooks wrote to (campaign_trapezoid / phase5_verify_blade). No edit
# needed unless your Drive layout differs.
SHARED_DIR   = DRIVE_ROOT / "campaign_trapezoid"                      # the campaign shards
VERIFICATION = DRIVE_ROOT / "stage3_verify_blade" / "verification.json"  # Stage-3.C fine J_fan
OUT_DIR      = DRIVE_ROOT / "stage4_blade_to"                         # carved fields + summary (Drive)
TOP_K        = 10          # TO the top-10 by fine J_fan (then pick 3 to print)
MAX_ITERS    = 40

# --- fidelity (measured on this blade; RAM + time are PER single FEA design) -----------------
# mesh 1.5mm:  65k tets |  ~5s/iter  | ~3 GB   | baseline (coarse internal structure)
# mesh 1.0mm: 180k tets | ~63s/iter  | ~6 GB   | ~42 min/volfrac
# mesh 0.8mm: 356k tets | ~2-3min/it | ~15 GB  | ~1.5-2 h/volfrac
# mesh 0.6mm: 690k tets | ~5-8min/it | ~30 GB  | ~4-5 h/volfrac  <-- finest; DEFAULT (best internal detail)
MESH_SIZE_M      = 0.6e-3            # finest -> best internal-beam resolution + thinnest skin
SKIN_THICKNESS_M = 0.5e-3            # solid shell kept on the air-pushing faces (~0.8x mesh; printable)

# Parallelism across designs (they are independent). RAM-BOUND, not core-bound: each worker holds a
# full factorization (~30 GB at 0.6mm, ~15 GB at 0.8mm), so N_WORKERS ~= session_RAM / per_design_RAM.
# GPU does NOT help — the solver is a CPU sparse direct factorization. Set this to your RAM budget.
N_WORKERS      = 5                          # e.g. 1 on a 50 GB session at 0.6mm; raise on a high-RAM box

SCREEN         = True                       # find max-removal-that-passes per design (vs fixed volfrac)
VOLFRAC_LADDER = DEFAULT_VOLFRAC_LADDER      # (0.25, 0.35, 0.45, 0.60) most-aggressive first
U_TIP_LIMIT_MM = 1.0                         # rigid-blade tip-deflection screen
STRESS_FOS     = 2.0                         # safety factor on PETG weak-axis yield for the stress screen

assert SHARED_DIR.exists(), f"campaign folder not found: {SHARED_DIR}"
assert VERIFICATION.exists(), f"verification.json not found: {VERIFICATION} (run Stage-3.C first)"
print(f"campaign: {SHARED_DIR}")
print(f"verify:   {VERIFICATION}")
print(f"top-{TOP_K} | mesh {MESH_SIZE_M*1e3:.1f} mm | skin {SKIN_THICKNESS_M*1e3:.2f} mm | "
      f"{N_WORKERS} worker(s) | screen={SCREEN} FOS={STRESS_FOS} | out {OUT_DIR}")

## 4. Preview — the top-10 designs that will be TO'd (by fine J_fan)

In [ ]:
from fanopt.cfd.blade_verify import top_verified_designs

designs = top_verified_designs(SHARED_DIR, VERIFICATION, top_k=TOP_K)
print(f"{len(designs)} designs to optimize (best fine J_fan first):")
for name, params, j3d in designs:
    print(f"  {name}   J_fan_3d={j3d:.4g}   rib_mode={'uniform' if params.uniform else 'ribbed'}")

## 5. RUN the per-design 3D TO
Writes `<name>_density.npy` + `summary.json` under `OUT_DIR` (on Drive). A failed design is
recorded with an `error` and skipped — one bad blade never aborts the batch.

In [ ]:
import run_phase2_blade_to

summary = run_phase2_blade_to.run(
    shared_dir=SHARED_DIR,
    verification=VERIFICATION,
    out_dir=OUT_DIR,
    top_k=TOP_K,
    max_iters=MAX_ITERS,
    mesh_size_m=MESH_SIZE_M,
    skin_thickness_m=SKIN_THICKNESS_M,
    n_workers=N_WORKERS,
    screen=SCREEN,
    volfrac_ladder=VOLFRAC_LADDER,
    u_tip_limit_m=U_TIP_LIMIT_MM / 1e3,
    stress_fos=STRESS_FOS,
    progress=True,
)
n_pass = sum(1 for r in summary["designs"] if r.get("screen_passed"))
print(f"\n{summary['n_succeeded']}/{summary['n_designs']} designs TO'd, "
      f"{n_pass} passed the screen -> {OUT_DIR}/summary.json")

## 6. Results — mass removed + structural screen per design

In [ ]:
import json
from fanopt.geometry.schema import SIGMA_Y_PETG_Z_PA

summ = json.loads((OUT_DIR / "summary.json").read_text())
rows = [r for r in summ["designs"] if "error" not in r]
rows.sort(key=lambda r: r["volume_removed_frac"], reverse=True)
yield_mpa = SIGMA_Y_PETG_Z_PA / 1e6

print(f"{'name':22} {'volfrac':>7} {'removed%':>8} {'mass_g':>7} {'u_tip_mm':>9} {'VM_MPa':>7} {'screen':>8}")
for r in rows:
    vf = r.get("accepted_volfrac")
    passed = r.get("screen_passed")
    # screened runs carry screen_passed; a fixed run falls back to the plain limits.
    label = ("PASS" if passed else "FAIL") if passed is not None else (
        "PASS" if (r["u_tip_max_mm"] < 1.0 and r["max_von_mises_mpa"] < yield_mpa) else "CHECK")
    print(f"{r['name']:22} {('%.2f'%vf) if vf is not None else '  -  ':>7} "
          f"{r['volume_removed_frac']*100:7.1f} {r['mass_kg']*1e3:7.1f} "
          f"{r['u_tip_max_mm']:9.3f} {r['max_von_mises_mpa']:7.2f} {label:>8}")
for r in summ["designs"]:
    if "error" in r:
        print(f"  [error] {r['name']}: {r['error']}")
print(f"\nscreen = worst tip deflection < {U_TIP_LIMIT_MM:.1f} mm AND worst von Mises < "
      f"{yield_mpa/STRESS_FOS:.0f} MPa (yield {yield_mpa:.0f} / FOS {STRESS_FOS:g}), over all 4 load cases.")
print("A FAIL means even the safest volfrac tried could not meet the screen — that design needs")
print("more material (raise the ladder) or is structurally marginal. Binding cert = §59.5 gate.")

## 7. Render carved vs solid — top designs in 3D
Loads the element centroids saved alongside each density field (no re-meshing, so the
colouring is guaranteed to align) and shows retained material — solid shell + retained core.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_SHOW = min(3, len(designs))
fig = make_subplots(rows=1, cols=N_SHOW, specs=[[{"type": "scene"}] * N_SHOW],
                    subplot_titles=[d[0][:14] for d in designs[:N_SHOW]])
for col, (name, _params, _j) in enumerate(designs[:N_SHOW], start=1):
    dens = np.load(OUT_DIR / f"{name}_density.npy")
    cen = np.load(OUT_DIR / f"{name}_centroids.npy")   # saved by the batch -> aligned to dens
    keep = dens > 0.5                                    # material retained after TO
    fig.add_trace(go.Scatter3d(
        x=cen[keep, 0], y=cen[keep, 1], z=cen[keep, 2], mode="markers",
        marker=dict(size=1.5, color=dens[keep], colorscale="Viridis", cmin=0.5, cmax=1.0),
        showlegend=False), row=1, col=col)
fig.update_layout(height=460, width=320 * N_SHOW, title="Retained material (density > 0.5)")
fig.show()